# 01 - Data Understanding and Data Cleaning
## Customer Feedback Intelligence System

### Objectives:
- Load raw e-commerce customer reviews dataset.
- Inspect structure, column data types, and missing values.
- Handle missing reviews and incomplete categorical entries.
- Remove redundant identifiers and duplicate records.
- Export the sanitized dataset to `data/processed/cleaned_reviews.csv`.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Define directory paths
RAW_DATA_PATH = Path("../data/raw/Womens Clothing E-Commerce Reviews.csv")
PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH = PROCESSED_DATA_DIR / "cleaned_reviews.csv"

In [2]:
# 1. Load raw dataset
df = pd.read_csv(RAW_DATA_PATH)
print("Initial Dataset Shape:", df.shape)
df.head()

Initial Dataset Shape: (23486, 11)


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [3]:
# 2. Inspect dataset structure and schema
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Unnamed: 0               23486 non-null  int64
 1   Clothing ID              23486 non-null  int64
 2   Age                      23486 non-null  int64
 3   Title                    19676 non-null  str  
 4   Review Text              22641 non-null  str  
 5   Rating                   23486 non-null  int64
 6   Recommended IND          23486 non-null  int64
 7   Positive Feedback Count  23486 non-null  int64
 8   Division Name            23472 non-null  str  
 9   Department Name          23472 non-null  str  
 10  Class Name               23472 non-null  str  
dtypes: int64(6), str(5)
memory usage: 9.5 MB


In [4]:
# 3. Check for missing values across all columns
missing_df = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Ratio (%)": (df.isnull().sum() / len(df)) * 100
}).sort_values(by="Missing Count", ascending=False)

missing_df[missing_df["Missing Count"] > 0]

,Missing Count,Missing Ratio (%)
Title,3810,16.222430
Review Text,845,3.597888
Department Name,14,0.059610
Class Name,14,0.059610
Division Name,14,0.059610


In [5]:
# 4. Remove unnecessary index columns if present
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)
    print("Dropped 'Unnamed: 0' column successfully.")

# 5. Handle missing review text
initial_rows = len(df)
df = df.dropna(subset=["Review Text"]).copy()

# Strip any leading/trailing whitespace and ensure non-empty strings
df["Review Text"] = df["Review Text"].astype(str).str.strip()
df = df[df["Review Text"] != ""].copy()
print(f"Rows dropped due to missing/empty Review Text: {initial_rows - len(df)}")

Dropped 'Unnamed: 0' column successfully.
Rows dropped due to missing/empty Review Text: 845


In [6]:
# 6. Handle missing categorical product attributes
category_cols = ["Division Name", "Department Name", "Class Name"]
initial_rows = len(df)
df = df.dropna(subset=category_cols).copy()
print(f"Rows dropped due to missing category metadata: {initial_rows - len(df)}")

# Fill missing Title values with empty string
df["Title"] = df["Title"].fillna("").astype(str).str.strip()

Rows dropped due to missing category metadata: 13


In [7]:
# 7. Check and drop duplicate records
full_duplicates = df.duplicated().sum()
text_duplicates = df.duplicated(subset=["Review Text", "Clothing ID"]).sum()
print(f"Full-row duplicates: {full_duplicates}")
print(f"Duplicate reviews for same product: {text_duplicates}")

df.drop_duplicates(subset=["Review Text", "Clothing ID"], inplace=True)
print("Final Cleaned Dataset Shape:", df.shape)

Full-row duplicates: 1
Duplicate reviews for same product: 2
Final Cleaned Dataset Shape: (22626, 10)


In [8]:
# 8. Export cleaned dataset
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Cleaned dataset successfully saved to: {PROCESSED_DATA_PATH}")

Cleaned dataset successfully saved to: ..\data\processed\cleaned_reviews.csv
